# react-lm: Steps 3–6 (Chat Template → QLoRA → Eval → GGUF)

Fine-tune **Qwen2.5-Coder-3B-Instruct** on the React golden dataset from [react-lm](https://github.com/your-org/react-lm).

**Prerequisites (local)**
- Run `python check_step2_ready.py` — Step 2 must pass.
- Produce `train.jsonl` and `eval.jsonl` (via `python build_dataset.py`).

**Prerequisites (Colab)**
- Runtime: **GPU** (T4 is enough).
- Provide both JSONL files using **one** of:
  1. **Clone** — set `REACT_LM_REPO` to your git URL in the data cell (or edit the default), then run that cell.
  2. **Upload** — when prompted, upload `train.jsonl` and `eval.jsonl`.
  3. **Notebook path** — open from a cloned repo so both files are already on disk.

Step 3 applies Unsloth's Qwen2.5 chat template via `apply_chat_template()` — do **not** hand-format `<|im_start|>` tokens.

Step 7 (Ollama + `Modelfile`) runs **locally** after you download the GGUF from Step 6.

## 1. Install Unsloth (GPU required)

In [ ]:
%%capture
import os, re, subprocess
if "COLAB_RELEASE_TAG" in os.environ:
    subprocess.run(
        "pip install -q unsloth datasets transformers trl accelerate bitsandbytes",
        shell=True,
        check=False,
    )
else:
    print("Local run: pip install unsloth datasets transformers trl accelerate bitsandbytes")

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get `train.jsonl` and `eval.jsonl`

`train.jsonl` is used for fine-tuning. `eval.jsonl` is held out for Step 5 only — never add it to the training map.

In [ ]:
from pathlib import Path
import os
import subprocess

TRAIN_PATH = Path("train.jsonl")
EVAL_PATH = Path("eval.jsonl")
# Set REACT_LM_REPO to your git remote, or edit the default below.
REPO_URL = os.environ.get(
    "REACT_LM_REPO",
    "https://github.com/your-org/react-lm.git",
)


def _describe(path: Path) -> str:
    return f"{path.resolve()} ({path.stat().st_size // 1024} KiB)"


def _clone_repo(url: str) -> None:
    dest = Path("react-lm")
    if dest.is_dir():
        print(f"Repo dir exists: {dest.resolve()}")
    else:
        print(f"Cloning {url} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", url, str(dest)],
            check=True,
        )
    for name in ("train.jsonl", "eval.jsonl"):
        src = dest / name
        if src.is_file() and not Path(name).is_file():
            Path(name).write_bytes(src.read_bytes())
            print(f"Copied {name} from clone")


def _ensure_jsonl() -> None:
    missing = [p.name for p in (TRAIN_PATH, EVAL_PATH) if not p.is_file()]
    if not missing:
        return
    if REPO_URL and "your-org" not in REPO_URL:
        _clone_repo(REPO_URL)
        missing = [p.name for p in (TRAIN_PATH, EVAL_PATH) if not p.is_file()]
    if missing and os.environ.get("COLAB_RELEASE_TAG"):
        print(f"Upload: {', '.join(missing)}")
        from google.colab import files

        uploaded = files.upload()
        for name in missing:
            if name not in uploaded:
                raise FileNotFoundError(f"Upload a file named {name}")
        missing = [p.name for p in (TRAIN_PATH, EVAL_PATH) if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing: {', '.join(missing)}. "
            "Set REACT_LM_REPO, upload both files in Colab, or run from the repo root."
        )


_ensure_jsonl()
print("train:", _describe(TRAIN_PATH))
print("eval:", _describe(EVAL_PATH))

## 3. Load model + Qwen2.5 chat template (Step 3)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

print("Model:", MODEL_NAME)
print("Chat template: qwen-2.5")

## 4. Map `conversations` → `text` (Step 3)

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files=str(TRAIN_PATH), split="train")
print("Rows loaded:", len(raw))
assert "conversations" in raw.column_names, "Expected conversations column in train.jsonl"


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}


dataset = raw.map(formatting_prompts_func, batched=True, remove_columns=raw.column_names)
print("Mapped to text column:", dataset)

## 5. Verify formatted sample (Step 3)

In [ ]:
sample_text = dataset[0]["text"]
print(sample_text[:2000])
print("\n---")
print("rows:", len(dataset))

for marker in ("<|im_start|>system", "<|im_start|>user", "<|im_start|>assistant"):
    assert marker in sample_text, f"Missing {marker} in formatted text"

lengths = []
for i in range(min(20, len(dataset))):
    enc = tokenizer(dataset[i]["text"], return_length=True)
    lengths.append(int(enc["length"][0]))

print(f"Token lengths (first {len(lengths)} rows): min={min(lengths)} max={max(lengths)} mean={sum(lengths)/len(lengths):.0f}")
over = sum(1 for L in lengths if L > MAX_SEQ_LENGTH)
if over:
    print(f"WARNING: {over} of {len(lengths)} sampled rows exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")
else:
    print(f"OK: sampled rows are under {MAX_SEQ_LENGTH} tokens")

## 6. QLoRA adapters (Step 4)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

## 7. Train with SFTTrainer (Step 4)

Adjust `num_train_epochs`, `max_steps`, and batch size for your GPU. Full run is typically **2–5 hours** on a T4.

For a quick smoke test during development, uncomment the subset line below.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "react-expert-lora"

# dataset = dataset.select(range(50))  # uncomment for a fast smoke test

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 8. Save LoRA adapter (Step 4)

Always run this after training so you can re-run eval or export without retraining.

In [ ]:
SAVE_DIR = "react-expert"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved LoRA adapter to {SAVE_DIR}/")

## 9. Evaluate on held-out `eval.jsonl` (Step 5)

Review **20–30** prompts the model never saw during training. Before exporting GGUF, check manually:

- Does it add `'use client'` when the prompt does not need interactivity?
- Does it use class components, `extends Component`, or `componentDidMount`?
- Is TypeScript reasonably typed (no trivial `any`)?

The heuristic flags below are **advisory** — read the generated code yourself.

In [ ]:
import re
from datasets import load_dataset
from unsloth import FastLanguageModel

EVAL_SAMPLE_SIZE = 25
MAX_NEW_TOKENS = 2048
GENERATION_TEMPERATURE = 0.2

INTERACTIVE_KEYWORDS = (
    "onclick", "onchange", "onsubmit", "onkeydown", "onkeyup",
    "usestate", "usereducer", "useref", "useeffect",
    "input", "button", "form", "modal", "dialog", "dropdown",
    "checkbox", "select", "toggle", "drag", "hover", "scroll",
    "keyboard", "click", "interactive", "client component",
    "animation", "framer", "gsap",
)

eval_raw = load_dataset("json", data_files=str(EVAL_PATH), split="train")
assert "conversations" in eval_raw.column_names
print(f"Eval rows: {len(eval_raw)} (sampling {min(EVAL_SAMPLE_SIZE, len(eval_raw))})")

FastLanguageModel.for_inference(model)


def require_message(convo, role: str) -> str:
    for m in convo:
        if m.get("role") == role:
            content = m.get("content")
            if content is not None:
                return content
    roles = [m.get("role") for m in convo]
    raise ValueError(
        f"Malformed eval row: missing '{role}' message with content. "
        f"Roles present: {roles}"
    )


def prompt_messages(convo):
    return [m for m in convo if m.get("role") != "assistant"]


def generate_reply(convo) -> str:
    messages = prompt_messages(convo)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    if hasattr(inputs, "items"):
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_len = inputs["input_ids"].shape[-1]
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=GENERATION_TEMPERATURE,
            use_cache=True,
        )
    else:
        inputs = inputs.to(model.device)
        input_len = inputs.shape[-1]
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=GENERATION_TEMPERATURE,
            use_cache=True,
        )
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


def heuristic_flags(user: str, code: str) -> list[str]:
    flags = []
    user_l = user.lower()
    code_l = code.lower()
    if "'use client'" in code or '"use client"' in code:
        if not any(kw in user_l for kw in INTERACTIVE_KEYWORDS):
            flags.append("use_client_without_interactive_prompt")
    if re.search(r"\bclass\s+\w+", code) or "extends component" in code_l:
        flags.append("class_component")
    if "componentdidmount" in code_l:
        flags.append("legacy_lifecycle")
    if re.search(r":\s*any\b|\bas any\b|<any>", code):
        flags.append("trivial_any")
    return flags

In [ ]:
sample_n = min(EVAL_SAMPLE_SIZE, len(eval_raw))
results = []

for idx in range(sample_n):
    convo = eval_raw[idx]["conversations"]
    try:
        user = require_message(convo, "user")
        reference = require_message(convo, "assistant")
    except ValueError as err:
        print("=" * 72)
        print(f"[{idx + 1}/{sample_n}] SKIP row {idx}: {err}")
        continue
    generated = generate_reply(convo)
    flags = heuristic_flags(user, generated)
    results.append({"idx": idx, "flags": flags, "user": user, "generated": generated, "reference": reference})

    print("=" * 72)
    print(f"[{idx + 1}/{sample_n}] USER:\n{user[:500]}{'...' if len(user) > 500 else ''}")
    print(f"\nFLAGS: {flags or 'none'}")
    print(f"\nGENERATED (first 1200 chars):\n{generated[:1200]}")
    print(f"\nREFERENCE (first 600 chars):\n{reference[:600]}...")

flagged = sum(1 for r in results if r["flags"])
passed = sample_n - flagged
print("\n" + "=" * 72)
print(f"Heuristic summary: passed={passed} flagged={flagged} total={sample_n}")
print("Review outputs above before setting PROCEED_TO_EXPORT = True in the next section.")

### Export gate

Set `PROCEED_TO_EXPORT = True` only after you are satisfied with Step 5 outputs.

In [ ]:
PROCEED_TO_EXPORT = False  # flip to True after manual review

if not PROCEED_TO_EXPORT:
    print("Step 6 skipped — set PROCEED_TO_EXPORT = True when ready.")
else:
    print("Ready for Step 6 GGUF export.")

## 10. Quantize and export GGUF (Step 6)

Exports **Q4_K_M** (~1.9 GB for this 3B model). Requires `PROCEED_TO_EXPORT = True` above.

Unsloth API: `save_pretrained_gguf(path, tokenizer, quantization_method=...)`. GGUF files are written under `{SAVE_DIR}_gguf/` (not the notebook cwd).

In [ ]:
from pathlib import Path

GGUF_QUANT = "q4_k_m"
GGUF_DIR = Path(f"{SAVE_DIR}_gguf")  # Unsloth default output directory


def find_gguf_paths(save_dir: str) -> list[Path]:
    """Unsloth writes *.gguf under {save_dir}_gguf/, not react-expert.Q4_K_M.gguf in cwd."""
    found: list[Path] = []
    for root in (Path(f"{save_dir}_gguf"), Path(save_dir)):
        if root.is_dir():
            found.extend(sorted(root.glob("*.gguf")))
    return found


if not PROCEED_TO_EXPORT:
    raise RuntimeError("Set PROCEED_TO_EXPORT = True in the gate cell after Step 5 review.")

model.save_pretrained_gguf(SAVE_DIR, tokenizer, quantization_method=GGUF_QUANT)

GGUF_PATHS = find_gguf_paths(SAVE_DIR)
if not GGUF_PATHS:
    raise FileNotFoundError(
        f"No .gguf files under {GGUF_DIR}/ or {SAVE_DIR}/. "
        "Check Unsloth export logs above for the actual path."
    )

print(f"GGUF export done ({len(GGUF_PATHS)} file(s)):")
for p in GGUF_PATHS:
    print(f"  {p.resolve()} ({p.stat().st_size // (1024 * 1024)} MiB)")

## 11. Download GGUF + deploy with Ollama (Step 7, local)

1. Download the `.gguf` file from Colab (cell below).
2. Place it next to the repo [`Modelfile`](../Modelfile) (or update the `FROM` path).
3. On your machine:

```bash
ollama create react-expert -f Modelfile
ollama run react-expert
```

In [ ]:
import os

if not PROCEED_TO_EXPORT:
    print("Nothing to download — complete Step 5 and set PROCEED_TO_EXPORT = True.")
else:
    from pathlib import Path

    if "find_gguf_paths" not in globals():
        def find_gguf_paths(save_dir: str) -> list[Path]:
            found: list[Path] = []
            for root in (Path(f"{save_dir}_gguf"), Path(save_dir)):
                if root.is_dir():
                    found.extend(sorted(root.glob("*.gguf")))
            return found

    paths = GGUF_PATHS if "GGUF_PATHS" in globals() and GGUF_PATHS else find_gguf_paths(SAVE_DIR)
    if not paths:
        raise FileNotFoundError(
            f"No .gguf under {SAVE_DIR}_gguf/ or {SAVE_DIR}/. Re-run the Step 6 export cell."
        )
    for p in paths:
        print(f"Download: {p} ({p.stat().st_size // (1024 * 1024)} MiB)")
    if os.environ.get("COLAB_RELEASE_TAG"):
        from google.colab import files

        for p in paths:
            files.download(str(p))
    else:
        print("Local run: copy the GGUF path above into your react-lm repo for Ollama.")